# 02 - Event loop JSON-lines (T3, issue #4)

Dwell-time state machine over the posture score with the frozen four-event
stdout contract (`bad_posture`, `recovered`, `heartbeat`, `error`).
Pure and dt-injectable: `update(state, score, dt)` accrues seconds, never
frames, so FPS jitter is absorbed. `None` scores freeze the dwell clocks
(hold last state); the heartbeat clock accrues regardless. stdout carries
only frozen events. No persistence (ADR-0002).

Consumes from T2: `select_largest_person`, `is_upright`, `is_recovered`.
Thresholds below mirror 01 (`UPRIGHT_SCORE`/`RECOVERED_SCORE` consistency is
test-pinned); unified into `engine.py` in T4, where the T1 `emit_error` /
T3 `emit_event` duplication is also resolved.

Live wiring (manual run; T4 hardens it): per `model(source=0, stream=True)`
frame, convert torch keypoints to numpy, `select_largest_person`,
`posture_score(xyn, vis, aspect=W/H from frame.shape)`, wall-clock dt,
`update`, `emit_event` each returned event; wrap the frame body so any
failure becomes the frozen `error` event, never a silent crash.

In [ ]:
import json

In [ ]:
UPRIGHT_SCORE = 70
RECOVERED_SCORE = 80
BAD_DWELL_S = 10
RECOVERED_DWELL_S = 3
HEARTBEAT_S = 5


def new_state():
    """Fresh dwell machine: not bad, clocks at zero, no score yet."""
    return {
        "in_bad": False,
        "outside_s": 0.0,
        "recovery_s": 0.0,
        "hb_s": 0.0,
        "last_score": None,
    }


def _is_upright(score):
    """Mirrors T2 is_upright; consistency test-pinned."""
    return score is not None and score >= UPRIGHT_SCORE


def _is_recovered(score):
    """Mirrors T2 is_recovered; consistency test-pinned."""
    return score is not None and score >= RECOVERED_SCORE


def update(state, score, dt):
    """Advance the machine by dt seconds; return emitted events in order."""
    events = []
    if score is not None:
        state["last_score"] = score
        if not state["in_bad"]:
            if _is_upright(score):
                state["outside_s"] = 0.0
            else:
                state["outside_s"] += dt
                if state["outside_s"] >= BAD_DWELL_S:
                    state["in_bad"] = True
                    state["recovery_s"] = 0.0
                    events.append(
                        {
                            "event": "bad_posture",
                            "score": score,
                            "dwell_s": state["outside_s"],
                        }
                    )
        elif _is_recovered(score):
            state["recovery_s"] += dt
            if state["recovery_s"] >= RECOVERED_DWELL_S:
                state["in_bad"] = False
                state["outside_s"] = 0.0
                state["recovery_s"] = 0.0
                events.append(
                    {"event": "recovered", "score": score, "dwell_s": RECOVERED_DWELL_S}
                )
        else:
            state["recovery_s"] = 0.0
    state["hb_s"] += dt
    while state["hb_s"] >= HEARTBEAT_S:
        state["hb_s"] -= HEARTBEAT_S
        events.append({"event": "heartbeat", "score": state["last_score"]})
    return events


def emit_event(event):
    """Print exactly one frozen JSON line on stdout."""
    print(json.dumps(event), flush=True)
    return event


def safe_update(state, score_fn, dt, on_error=None):
    """Run one frame's score_fn; any failure becomes the error event.

    on_error(code, message) fires after emission so the loop owner can
    stop the loop; default None preserves emit-and-continue.
    """
    try:
        return update(state, score_fn(), dt)
    except Exception as exc:
        code = getattr(exc, "code", "UPDATE_FAILED")
        message = str(exc) or type(exc).__name__
        emit_event({"event": "error", "code": code, "message": message})
        if on_error is not None:
            on_error(code, message)
        return []